[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TianshuangQiu/TorchCode/blob/master/templates/57_sat_overlap.ipynb)

# 🟡 Medium: SAT Convex Polygon Overlap

Given two convex polygons `poly_a` and `poly_b` with vertices listed in counter-clockwise order, return `True` if they overlap (share any interior or boundary point).

Use the **Separating Axis Theorem (SAT)**: two convex shapes do *not* overlap if and only if there exists an axis along which their projections are disjoint. For convex polygons, the only axes you need to test are the edge normals of both polygons.

### Signature
```python
def sat_overlap(poly_a: np.ndarray, poly_b: np.ndarray) -> bool:
    # poly_a: (V, 2) float — convex polygon, CCW vertex order
    # poly_b: (W, 2) float — convex polygon, CCW vertex order
    # returns: bool
```

### Rules
- Do **NOT** use Python `for` loops over individual vertices
- Use `np.roll` to get cyclic edge vectors; vectorise projections with `@`
- Polygons sharing only a boundary point count as overlapping

### Example
```
square = [[0,0],[1,0],[1,1],[0,1]]
tri    = [[0.5,0.5],[1.5,0.5],[1.0,1.5]]   # overlaps upper-right corner
output: True

tri2   = [[2,0],[3,0],[2.5,1]]              # clearly separated
output: False
```

> **Reduction step (say this before coding):** `overlap` is True iff for every edge normal of `poly_a` and every edge normal of `poly_b`, the interval `[min(poly_a @ axis), max(poly_a @ axis)]` and `[min(poly_b @ axis), max(poly_b @ axis)]` overlap — equivalently, no axis separates the two projection intervals.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sat_overlap(poly_a, poly_b):
    # poly_a: (V, 2) — convex polygon vertices CCW
    # poly_b: (W, 2) — convex polygon vertices CCW
    # returns: bool
    pass  # Replace this

In [ ]:
# 🧪 Test your implementation
square = np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0]])
tri_in  = np.array([[0.5,0.5],[1.5,0.5],[1.0,1.5]])  # overlaps corner
tri_out = np.array([[2.0,0.0],[3.0,0.0],[2.5,1.0]])   # clearly separated
print("overlapping:", sat_overlap(square, tri_in))   # expect True
print("separated:  ", sat_overlap(square, tri_out))  # expect False

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: identical squares ─────────────────────────────────────────────
sq = np.array([[0.0,0.0],[1.0,0.0],[1.0,1.0],[0.0,1.0]])
assert sat_overlap(sq, sq.copy()) == True, "Identical squares → overlap"
print("Test 1 passed: identical squares")

# ── Test 2: non-overlapping squares ───────────────────────────────────────
sq2 = np.array([[5.0,0.0],[6.0,0.0],[6.0,1.0],[5.0,1.0]])
assert sat_overlap(sq, sq2) == False, "Separated on X → no overlap"
sq3 = np.array([[0.0,5.0],[1.0,5.0],[1.0,6.0],[0.0,6.0]])
assert sat_overlap(sq, sq3) == False, "Separated on Y → no overlap"
print("Test 2 passed: non-overlapping squares")

# ── Test 3: half-overlapping squares ─────────────────────────────────────
a = np.array([[0.0,0.0],[2.0,0.0],[2.0,2.0],[0.0,2.0]])
b = np.array([[1.0,0.0],[3.0,0.0],[3.0,2.0],[1.0,2.0]])
assert sat_overlap(a, b) == True, "Half-overlapping → overlap"
print("Test 3 passed: half-overlapping squares")

# ── Test 4: triangle vs square ────────────────────────────────────────────
square4 = np.array([[0.0,0.0],[4.0,0.0],[4.0,4.0],[0.0,4.0]])
tri_in  = np.array([[1.0,1.0],[3.0,1.0],[2.0,3.0]])
tri_out = np.array([[4.01,0.0],[6.0,0.0],[5.0,2.0]])
assert sat_overlap(square4, tri_in) == True,  "Triangle inside square → overlap"
assert sat_overlap(square4, tri_out) == False, "Triangle just outside → no overlap"
print("Test 4 passed: triangle vs square")

# ── Test 5: symmetry + timing on 100 random pairs ─────────────────────────
def rand_convex(rng, n=5, s=3.0):
    angles = np.sort(rng.uniform(0, 2*np.pi, n))
    r = rng.uniform(0.5, s, n)
    cx, cy = rng.uniform(0, 10), rng.uniform(0, 10)
    return np.stack([cx + r*np.cos(angles), cy + r*np.sin(angles)], axis=1)

rng = np.random.default_rng(99)
t0 = time.time()
for _ in range(100):
    pa, pb = rand_convex(rng), rand_convex(rng)
    r1, r2 = sat_overlap(pa, pb), sat_overlap(pb, pa)
    assert r1 == r2, f"Not symmetric: {r1} vs {r2}"
elapsed = time.time() - t0
assert elapsed < 3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: 100 random pairs, symmetric ({elapsed:.3f}s)")

print("\nAll tests passed!")